In [23]:
import pandas as pd

In [24]:
df=pd.read_csv("../data/black_friday_dataset.csv")
print(f"missing values")
print(df.isnull().sum())


missing values
User_ID                            0
Product_ID                         0
Gender                             0
Age                                0
Occupation                         0
City_Category                      0
Stay_In_Current_City_Years         0
Marital_Status                     0
Product_Category_1                 0
Product_Category_2            173638
Product_Category_3            383247
Purchase                           0
dtype: int64


In [25]:
print(df['Purchase'].describe())

count    550068.000000
mean       9263.968713
std        5023.065394
min          12.000000
25%        5823.000000
50%        8047.000000
75%       12054.000000
max       23961.000000
Name: Purchase, dtype: float64


In [26]:
df_copy=df.copy()

In [27]:
# fill missing product categories with 0 as none
df_copy['Product_Category_2']=  df_copy['Product_Category_2'].fillna(0).astype(int)
df_copy['Product_Category_3']= df_copy['Product_Category_3'].fillna(0).astype(int)
print(df_copy.info())

<class 'pandas.DataFrame'>
RangeIndex: 550068 entries, 0 to 550067
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   User_ID                     550068 non-null  int64
 1   Product_ID                  550068 non-null  str  
 2   Gender                      550068 non-null  str  
 3   Age                         550068 non-null  str  
 4   Occupation                  550068 non-null  int64
 5   City_Category               550068 non-null  str  
 6   Stay_In_Current_City_Years  550068 non-null  str  
 7   Marital_Status              550068 non-null  int64
 8   Product_Category_1          550068 non-null  int64
 9   Product_Category_2          550068 non-null  int64
 10  Product_Category_3          550068 non-null  int64
 11  Purchase                    550068 non-null  int64
dtypes: int64(7), str(5)
memory usage: 50.4 MB
None


In [28]:
#integrity checks
purchase_numeric = pd.to_numeric(df_copy['Purchase'], errors='coerce')
invalid_purchases = df_copy[purchase_numeric.isna()]

if len(invalid_purchases) > 0:
    print(f"Found {len(invalid_purchases)} non-numeric rows in 'Purchase' column! Handling them...")

    df_copy = df_copy.dropna(subset=['Purchase'])
else:
    print("'Purchase' column integrity check passed: All values are clean numbers.")

'Purchase' column integrity check passed: All values are clean numbers.


In [29]:
# Validate 'User_ID' column string anomalies
user_id_numeric = pd.to_numeric(df_copy['User_ID'], errors='coerce')
invalid_users = df_copy[user_id_numeric.isna()]

if len(invalid_users) > 0:
    print(f"Found {len(invalid_users)} corrupted User_IDs! Removing them...")
    df_copy = df_copy.dropna(subset=['User_ID'])
else:
    print("'User_ID' column integrity check passed: All identifiers are valid.")

'User_ID' column integrity check passed: All identifiers are valid.


In [30]:
# 3. Standardize Categorical Text Casing (e.g., 'City_Category' or 'Gender')
# This fixes issues where 'A' and 'a' might be treated as different cities
if 'City_Category' in df_copy.columns:
    df_copy['City_Category'] = df_copy['City_Category'].astype(str).str.upper().str.strip()
    print("'City_Category' string values standardized to UPPERCASE and stripped of extra spaces.")

print("\nData schema validation complete. Dataset is structurally secure!")

'City_Category' string values standardized to UPPERCASE and stripped of extra spaces.

Data schema validation complete. Dataset is structurally secure!


In [31]:
# check and drop duplicate
duplicate_count = df_copy.duplicated().sum()
print(f"Duplicate rows found and removed: {duplicate_count}")
if duplicate_count > 0:
    df_copy = df_copy.drop_duplicates()

Duplicate rows found and removed: 0


In [32]:
# Behavioral Aggregation using the copy
customer_df = df_copy.groupby('User_ID').agg({
    'Product_ID': 'count',                 # Frequency: Total items bought
    'Purchase': 'sum',                     # Monetary: Total spend
    'Product_Category_1': 'nunique'        # Diversity: Distinct main categories explored
}).reset_index()

In [33]:
customer_df.columns = ['customer_id', 'frequency', 'monetary', 'category_diversity']

In [35]:
customer_df.to_csv("../data/customer_behavior_segments.csv", index=False)
print("Saved successfully!")

Saved successfully!


Scaling and clustering

In [36]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = ['frequency', 'monetary', 'category_diversity']

scaler = StandardScaler()
scaled_features = scaler.fit_transform(customer_df[features])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_df['cluster'] = kmeans.fit_predict(scaled_features)

cluster_profiles = customer_df.groupby('cluster').agg({
    'customer_id': 'count',
    'frequency': 'mean',
    'monetary': 'mean',
    'category_diversity': 'mean'
}).rename(columns={'customer_id': 'total_customers'}).reset_index()

print("--- CUSTOMER CLUSTER PROFILES ---")
print(cluster_profiles.to_string(index=False))

--- CUSTOMER CLUSTER PROFILES ---
 cluster  total_customers  frequency     monetary  category_diversity
       0             2717  29.780640 2.859239e+05            6.521531
       1              939 207.533546 1.906912e+06           14.119276
       2              260 456.619231 3.966360e+06           16.173077
       3             1975  78.764051 7.580322e+05           10.936709
